## Setup
- `code_source`: Source used to obtain workspace code (`github`, `google_drive`, `gdrive_relative`, `url`, `local_dir`).
- `code_path`: Repository URL, Google Drive folder, archive URL, or local path. Default: `https://github.com/Neutron84/universal-ai-workspace.git`.
- `workspace_mode`: Storage mode (`local_ssd` for drive-free execution, `hybrid`, `google_drive`, `gdrive_relative`, `custom_path`).
- `workspace_path`: Path relative to MyDrive for hybrid/relative modes, or absolute path for custom modes.
- `input_type`: Source format of input weights (`auto`, `huggingface`, `direct_upload`, `local_path`, `google_drive`, `gdrive_relative`, `gdrive_share`, `url`, `civitai`, `git_repo`).
- `input_path`: Model identifier, repo ID, URL, share ID, or filesystem path (ignored for direct upload).
- `input_file`: Optional specific file inside repo or folder (leave blank for automatic selection).
- `unpack_archive`: Automatically extracts input archives (.zip, .tar.gz, .tar.xz, .tar.bz2).
- `task`: Operation mode (`serve` to run OpenAI-compatible API server, `convert_and_quantize`, or `dashboard_only`).
- `engine`: Inference engine backend (`auto`, `llama.cpp`, `ik_llama`, `vllm`, `transformers`, `unsloth`, `exllamav2`, `diffusers`, `faster-whisper`).
- `quantizer`: Quantization backend (`auto`, `llama.cpp`, `ik_llama`, `unsloth`, `gptqmodel`, `autoawq`, `exllamav2`, `ct2`).
- `target_quant`: Target quantization format (leave as `auto` or select specific quant format).
- `context_size`: Context window size in tokens.
- `use_imatrix`: Enables importance matrix calibration for llama.cpp quantization.
- `output_type`: Destination for artifacts (`none`, `local_ssd`, `direct_download`, `google_drive`, `gdrive_relative`, `gdrive_direct`, `huggingface_hub`, `custom_path`).
- `output_path`: Destination path or Hugging Face repository ID (`username/repo-name`).
- `output_name`: Custom filename or folder name for exported artifacts (leave blank to auto-derive).
- `zip_output`: Compresses exported artifacts into a zip archive before export or download.
- `cleanup_source`: Deletes oversized source files after conversion to save disk space.
- `enable_tunnel`: Establishes a public Cloudflare tunnel endpoint.
- `enable_api_auth`: Generates and enforces an authorization token for API endpoints.
- `port`: Local port for the inference server.
- `hf_token`: Hugging Face access token for authenticated repositories.

After configuring the form below, run Runtime > Run All (Ctrl + F9) or run individual sections below.


In [ ]:
#@title Setup Configuration
#@markdown ---
#@markdown ### 1. Code & Workspace Location
code_source = "github" #@param ["github", "google_drive", "gdrive_relative", "url", "local_dir"]
code_path = "https://github.com/Neutron84/universal-ai-workspace.git" #@param {type:"string"}
workspace_mode = "local_ssd" #@param ["local_ssd", "hybrid", "google_drive", "gdrive_relative", "custom_path"]
workspace_path = "AI_Workspace" #@param {type:"string"}

#@markdown ---
#@markdown ### 2. Input Source & Location
input_type = "auto" #@param ["auto", "huggingface", "direct_upload", "local_path", "google_drive", "gdrive_relative", "gdrive_share", "url", "civitai", "git_repo"]
input_path = "Qwen/Qwen2.5-0.5B-Instruct-GGUF" #@param {type:"string"}
input_file = "" #@param {type:"string"}
unpack_archive = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 3. Task, Engine & Quantizer / Converter
task = "serve" #@param ["serve", "convert_and_quantize", "dashboard_only"]
engine = "auto" #@param ["auto", "llama.cpp", "ik_llama", "vllm", "transformers", "unsloth", "exllamav2", "diffusers", "faster-whisper"]
quantizer = "auto" #@param ["auto", "llama.cpp", "ik_llama", "unsloth", "gptqmodel", "autoawq", "exllamav2", "ct2"]
target_quant = "auto" #@param {type:"string"}
context_size = 4096 #@param {type:"integer"}
use_imatrix = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 4. Output Target & Location
output_type = "none" #@param ["none", "local_ssd", "direct_download", "google_drive", "gdrive_relative", "gdrive_direct", "huggingface_hub", "custom_path"]
output_path = "AI_Workspace/outputs" #@param {type:"string"}
output_name = "" #@param {type:"string"}
zip_output = False #@param {type:"boolean"}
cleanup_source = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 5. Runtime & Networking
enable_tunnel = True #@param {type:"boolean"}
enable_api_auth = True #@param {type:"boolean"}
port = 8000 #@param {type:"integer"}
hf_token = "" #@param {type:"string"}

import os
import shutil
import subprocess
import sys
from pathlib import Path

# 1. Mount Google Drive only if required by any configured mode
drive_required_modes = {"google_drive", "gdrive_relative", "gdrive_direct", "hybrid"}
needs_drive = any(
    str(val).strip().lower() in drive_required_modes
    for val in (code_source, workspace_mode, input_type, output_type)
)

if needs_drive:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            print("Mounting Google Drive to /content/drive...")
            drive.mount("/content/drive")
        else:
            print("Google Drive is already mounted.")
    except Exception as e:
        print(f"Note: Google Drive mount unavailable ({e}). Operating in local storage mode.")
else:
    print("Local storage mode active. Google Drive mount skipped.")

# 2. Acquire workspace source code into local environment
LOCAL_APP_DIR = Path("/content/app")
LOCAL_APP_DIR.mkdir(parents=True, exist_ok=True)

if code_source == "github":
    repo_url = code_path if (code_path.startswith("http") or code_path.startswith("git")) else f"https://github.com/{code_path}.git"
    if not (LOCAL_APP_DIR / "workspace_core").is_dir():
        print(f"Cloning repository {repo_url} into {LOCAL_APP_DIR}...")
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(LOCAL_APP_DIR)], check=True)
elif code_source in ["google_drive", "gdrive_relative"]:
    drive_src = Path("/content/drive/MyDrive") / code_path.strip().lstrip("/")
    if drive_src.is_dir() and (drive_src / "workspace_core").is_dir():
        print(f"Syncing workspace code from Google Drive to {LOCAL_APP_DIR}...")
        for item in ["workspace_core", "configs", "pyproject.toml", "README.md"]:
            s = drive_src / item
            d = LOCAL_APP_DIR / item
            if s.is_dir():
                shutil.copytree(str(s), str(d), dirs_exist_ok=True)
            elif s.is_file():
                shutil.copy2(str(s), str(d))
elif code_source == "url":
    print(f"Downloading source archive from {code_path}...")
    import urllib.request, zipfile
    tmp_zip = Path("/content/source_archive.zip")
    urllib.request.urlretrieve(code_path, str(tmp_zip))
    with zipfile.ZipFile(str(tmp_zip), "r") as z:
        z.extractall(str(LOCAL_APP_DIR))
elif code_source == "local_dir":
    loc_src = Path(code_path).resolve()
    if loc_src != LOCAL_APP_DIR.resolve() and (loc_src / "workspace_core").is_dir():
        for item in ["workspace_core", "configs", "pyproject.toml"]:
            s = loc_src / item
            if s.is_dir():
                shutil.copytree(str(s), str(LOCAL_APP_DIR / item), dirs_exist_ok=True)

# 3. Register python path
if str(LOCAL_APP_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_APP_DIR))
if Path("/content/universal-ai-workspace").is_dir() and "/content/universal-ai-workspace" not in sys.path:
    sys.path.insert(0, "/content/universal-ai-workspace")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

from workspace_core import get_config
config = get_config(workspace_mode=workspace_mode, workspace_path=workspace_path)
print(f"Workspace initialized. Mode: {config.mode.value} | Path: {config.drive_root}")


In [ ]:
#@title 1. Environment & Hardware Initialization
#@markdown Installs system dependencies, engine binaries, and verifies hardware.
from workspace_core import EnvironmentManager, get_hardware_profile

env_mgr = EnvironmentManager(config)
env_mgr.ensure_system_packages()

if enable_tunnel:
    env_mgr.fetch_cloudflared()

target_engine = engine.lower().strip()
target_quantizer = quantizer.lower().strip()

if target_engine in ("ik_llama", "ik_llama.cpp") or target_quantizer in ("ik_llama", "ik_llama.cpp"):
    env_mgr.fetch_ik_llama_cpp()
else:
    env_mgr.fetch_llama_cpp()

profile_to_install = "core"
if target_engine in ("transformers", "unsloth") or target_quantizer == "unsloth":
    profile_to_install = "unsloth" if (target_engine == "unsloth" or target_quantizer == "unsloth") else "transformers"
elif target_engine == "vllm":
    profile_to_install = "vllm"
elif target_engine == "diffusers":
    profile_to_install = "diffusion"
elif target_engine in ("audio", "faster-whisper") or target_quantizer == "ct2":
    profile_to_install = "audio"

env_mgr.install_profile(profile_to_install)

hw = get_hardware_profile()
gpu_desc = f"{hw.primary_gpu.name} ({hw.primary_gpu.total_vram_mb:.0f} MB VRAM, CC {hw.primary_gpu.compute_capability[0]}.{hw.primary_gpu.compute_capability[1]})" if hw.primary_gpu else "No GPU detected (CPU mode)"

print("System Diagnostics:")
print(f"  GPU:      {gpu_desc}")
print(f"  RAM:      {hw.ram_available_gb:.1f} GB available / {hw.ram_total_gb:.1f} GB total")
print(f"  SSD:      {hw.ssd_free_gb:.1f} GB free")
print(f"  CUDA:     {hw.cuda_version or N/A}")
print(f"  Python:   {hw.python_version}")


In [ ]:
#@title 2. Input Resolution & Download
#@markdown Resolves input source URI, downloads weights, and extracts archives if required.
from workspace_core import ModelInspector, Downloader, IOResolver

inspector = ModelInspector(hf_token=hf_token or None)
resolved_input_path = input_path

if input_type == "direct_upload":
    from google.colab import files
    print("Upload your model file or archive:")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        resolved_input_path = str(Path("/content") / uploaded_name)
    else:
        raise ValueError("No file uploaded.")

print(f"Resolving input: {resolved_input_path} (mode: {input_type})...")
resolved_input = IOResolver.resolve_input(
    uri=resolved_input_path,
    mode=input_type,
    download_dir=config.runtime_root / "inputs",
    hf_token=hf_token or None,
    unpack_archive=unpack_archive,
    input_file=input_file or None
)

print(f"Input resolved. Type: {resolved_input.resolved_type} | Identifier: {resolved_input.identifier}")


In [ ]:
#@title 3. Model Inspection & Capability Analysis
#@markdown Inspects architecture flags and calculates optimal engine and quantization fit.
from workspace_core import AutoSelector

insp_target = str(resolved_input.local_path) if resolved_input.local_path else resolved_input_path
insp = inspector.inspect(insp_target, input_mode=input_type)
if insp.error:
    raise RuntimeError(f"Inspection error: {insp.error}")

selector = AutoSelector(config=config)
rec = selector.recommend(
    inspection=insp,
    requested_ctx=context_size,
    override_quant=target_quant if target_quant != "auto" else None,
    override_engine=engine if engine != "auto" else None,
    override_quantizer=quantizer if quantizer != "auto" else None,
    task=task
)

print("Decision Summary:")
print(f"  {rec.decision_summary}")
print(f"  Selected Engine:    {rec.recommended_engine}")
print(f"  Selected Quantizer: {rec.recommended_converter or none}")
print(f"  Recommended Quant:  {rec.recommended_quant}")
print(f"  VRAM Budget:        {rec.vram_budget_mb:.0f} MB")


In [ ]:
#@title 4. Task Execution (Serve or Convert)
#@markdown Runs inference server, launches dashboard, or executes model conversion and quantization.
import time
from workspace_core import ProcessManager, ModelRegistry, Downloader, get_converter
from workspace_core.dashboard import launch_dashboard

downloader = Downloader(config)
registry = ModelRegistry(config)
pm = ProcessManager(config)

if task == "dashboard_only":
    print("Launching management dashboard on port 7860...")
    launch_dashboard(port=7860, share=True)

elif task == "convert_and_quantize":
    print("Preparing model files for conversion...")
    sync_to_drive = (workspace_mode in ["hybrid", "google_drive", "gdrive_relative"])
    dl_res = downloader.resolve_or_download(
        inspection=insp,
        variant=rec.selected_variant,
        sync_to_drive=sync_to_drive
    )
    source_model_file = dl_res.primary_file_path

    converter_name = rec.recommended_converter or "llama.cpp"
    print(f"Running conversion with {converter_name}...")
    converter = get_converter(converter_name, config)
    converted_path = converter.convert_or_quantize(
        input_path=source_model_file,
        target_quant=rec.recommended_quant,
        cleanup_source=cleanup_source,
        use_imatrix=use_imatrix
    )
    print(f"Conversion complete. Artifact saved to: {converted_path}")
    task_artifact_path = converted_path

else:  # task == "serve"
    sync_to_drive = (workspace_mode in ["hybrid", "google_drive", "gdrive_relative"])
    dl_res = downloader.resolve_or_download(
        inspection=insp,
        variant=rec.selected_variant,
        sync_to_drive=sync_to_drive
    )
    final_path = dl_res.primary_file_path

    if rec.needs_requant and rec.requant_target_quant:
        print(f"Running re-quantization to {rec.requant_target_quant}...")
        final_path = AutoSelector.execute_requantize(
            source_path=final_path,
            target_quant=rec.requant_target_quant,
            backup_to_drive=sync_to_drive,
            cleanup_source=cleanup_source
        )

    cat_map = {
        "llm-gguf": config.models_gguf_dir,
        "vlm-gguf": config.models_gguf_dir,
        "embedding-gguf": config.models_gguf_dir,
        "llm-safetensors": config.models_safetensors_dir,
        "diffusion": config.models_diffusion_dir,
        "audio": config.models_audio_dir,
    }
    dest_dir = cat_map.get(insp.model_type, config.models_gguf_dir)
    drive_path = str(dest_dir / final_path.name) if sync_to_drive else None
    shard_paths = [str(p) for p in dl_res.all_downloaded_files if p.resolve() != final_path.resolve()]

    entry = registry.register_model(
        model_id=dl_res.model_id,
        name=dl_res.model_id,
        type=insp.model_type,
        source_url=insp.raw_source,
        quantization=rec.recommended_quant,
        total_size_bytes=dl_res.total_size_bytes,
        local_path=str(final_path),
        drive_path=drive_path,
        is_synced_to_drive=sync_to_drive,
        mmproj_path=str(dl_res.mmproj_file_path) if dl_res.mmproj_file_path else None,
        shard_files=shard_paths,
        set_active=True
    )

    print(f"Starting server with engine {rec.recommended_engine} on port {port}...")
    pm.start_server(
        model_id=entry.model_id,
        port=port,
        enable_auth=enable_api_auth,
        recommendation=rec
    )

    if enable_tunnel:
        print("Launching Cloudflare tunnel...")
        pm.start_tunnel(port=port)

    state = pm.get_state()
    print("-" * 50)
    print(f"Server Status:   {Running if state.is_server_running else Failed}")
    print(f"Backend Engine:  {state.engine_name}")
    print(f"Local Endpoint:  http://127.0.0.1:{state.port}/v1")
    if state.public_url:
        print(f"Public Tunnel:   {state.public_url}/v1")
    if state.api_key:
        print(f"API Key:         {state.api_key}")
    print("-" * 50)
    task_artifact_path = final_path


In [ ]:
#@title 5. Artifact Export & Output Management
#@markdown Exports generated models or logs to Local SSD, Google Drive, Hugging Face Hub, or browser download.
from workspace_core import OutputManager

if output_type != "none" and "task_artifact_path" in locals() and task_artifact_path.exists():
    out_mgr = OutputManager(paths=config.paths, hf_token=hf_token or None)
    print(f"Exporting artifact {task_artifact_path.name} to {output_type}...")
    export_res = out_mgr.export_artifact(
        source_path=task_artifact_path,
        output_mode=output_type,
        destination=output_path,
        output_name=output_name or None,
        zip_output=zip_output,
        cleanup_source=cleanup_source,
    )
    print(f"Export Status: {export_res.get(message, Complete)}")
    if export_res.get("exported_path"):
        print(f"Location:      {export_res.get(exported_path)}")
else:
    print("Output export skipped (output_type=none or no artifact to export).")


In [ ]:
#@title 6. Verification & Client Integration
#@markdown Verifies running server endpoints with streaming request and outputs ready-to-use client snippets.
import json
import requests
from workspace_core import ProcessManager
from workspace_core.dashboard import WorkspaceDashboard

pm = ProcessManager(config)
state = pm.get_state()

if state.is_server_running:
    base_url = f"http://127.0.0.1:{state.port}"
    headers = {}
    if state.api_key:
        headers["Authorization"] = f"Bearer {state.api_key}"

    try:
        r = requests.get(f"{base_url}/v1/models", headers=headers, timeout=5)
        print("GET /v1/models response:")
        print(json.dumps(r.json(), indent=2))
    except Exception as e:
        print(f"Healthcheck error: {e}")

    print("\nTesting streaming completion:")
    payload = {
        "model": state.active_model or "default",
        "messages": [{"role": "user", "content": "Explain neural network weights in two sentences."}],
        "stream": True,
        "temperature": 0.2
    }
    try:
        r = requests.post(f"{base_url}/v1/chat/completions", json=payload, headers=headers, stream=True, timeout=30)
        for line in r.iter_lines():
            if line:
                dec = line.decode("utf-8")
                if dec.startswith("data: "):
                    data_str = dec[6:].strip()
                    if data_str == "[DONE]":
                        break
                    chunk = json.loads(data_str)
                    delta = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
                    print(delta, end="", flush=True)
        print("\n\nVerification completed successfully.")
    except Exception as e:
        print(f"Stream verification error: {e}")

    dash = WorkspaceDashboard(config)
    py_snippet, curl_snippet = dash.generate_code_snippets(state)
    print("\n" + "=" * 30 + " Python Client Example " + "=" * 30)
    print(py_snippet)
    print("=" * 30 + " cURL Client Example " + "=" * 30)
    print(curl_snippet)
else:
    print("Server is not currently running. Verification skipped.")


## Disclaimer

Google Colab provides access to shared computing resources subject to availability and usage limits. Free-tier GPU allocations are subject to timeout restrictions and preemption. Extended GPU usage or excessive memory consumption may result in session termination by the platform.


## License

This project is licensed under the MIT License. See the [LICENSE](LICENSE) file for details.
